# RS/OF fits on spikes vs dF/F, free locomotion vs treadmill

Does deconvolution bring the free-locomotion RS/OF fits closer to the treadmill fits?

For each ROI of the four motor sessions, the all-trial `gaussian_2d` fits (k=1) are compared:

| | data | response | fit file (`fit_rs_of_tuning_gaussian_2d_k1_*.pickle`) |
| :-- | :-- | :-- | :-- |
| **A** | free locomotion (`SpheresPermTubeReward`, closed loop), per frame | dF/F | `legacy_refit` |
| **B** | same | spikes | `legacy_refit_spks` |
| **T** | treadmill (`SpheresTubeMotor`), trial average, plateau, tight bounds | dF/F | `treadmill_trial_average_legacy_plateau` |
| **T′** | same | spikes | `treadmill_trial_average_legacy_plateau_spks` |

Spike fits were made with `precompute_data/fit_revision_treadmill.py --use-col spks` on NEMO
(run dir `rsof_spks_run`), with the spikes re-deconvolved on 2026-10-05 (τ 1.1 s for GCaMP6s / PZAG,
0.25 s for GCaMP6f / PZAH). Test R² comes from the k=5 fits of the same files.

Fits are read from the pickles, not from `neurons_df.pickle`: in PZAG16.3c `neurons_df` holds an older
free-locomotion fit, and on NEMO the treadmill columns are still the wide-bound fit. The local copy of the
T pickles is the tight-bound refit of 2026-08-28.

Neurons are selected on the treadmill only (depth tuned on the treadmill, T test R² above threshold, T
preference not on the tight bounds), so the selection favours neither A nor B.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
import flexiznam as flz

from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting import rsof_plots
from v1_depth_map.figure_utils.treadmill import compute_treadmill_rsof_bins
from v1_depth_map.precompute_data import fit_revision_treadmill as frt

PROJECT = frt.PROJECT
SESSIONS = frt.SESSIONS
FITS = {
    "A": "legacy_refit",
    "B": "legacy_refit_spks",
    "T": "treadmill_trial_average_legacy_plateau",
    "Tp": "treadmill_trial_average_legacy_plateau_spks",
}
LABELS = {
    "A": "free loco dF/F",
    "B": "free loco spikes",
    "T": "treadmill dF/F",
    "Tp": "treadmill spikes",
}
WIDE = frt.DEFAULT_PARAM_RANGE
TIGHT = frt.TREADMILL_PARAM_RANGE
R2_THR = 0.02

flm_sess = flz.get_flexilims_session(project_id=PROJECT)
save_folder = flz.get_processed_path(f"{PROJECT}/analysis/rsof_spikes_vs_dff")
save_folder.mkdir(parents=True, exist_ok=True)
print(f"Saving to {save_folder}")
mpl.rcParams["pdf.fonttype"] = 42
cm = 1 / 2.54

## Load the fits

In [ ]:
def session_folder(session):
    return pipeline_utils.create_neurons_ds(
        session_name=session, flexilims_session=flm_sess, project=PROJECT, conflicts="skip"
    ).path_full.parent


def load_fits(session):
    """Preferred RS (m/s), OF (deg/s), RS/OF (m) and k=5 test R² of every fit, one row per ROI."""
    folder = session_folder(session)
    ndf = pd.read_pickle(folder / "neurons_df.pickle")
    df = pd.DataFrame(
        dict(
            session=session,
            roi=ndf.roi.values,
            is_depth_neuron_treadmill=ndf.is_depth_neuron_treadmill.values.astype(bool),
        )
    )
    for key, tag in FITS.items():
        k1 = pd.read_pickle(folder / f"fit_rs_of_tuning_gaussian_2d_k1_{tag}.pickle")
        k5 = pd.read_pickle(folder / f"fit_rs_of_tuning_gaussian_2d_k5_{tag}.pickle")
        assert np.array_equal(k1.roi.values, df.roi.values)
        assert np.array_equal(k5.roi.values, df.roi.values)
        df[f"rs_{key}"] = k1.preferred_RS_closedloop_g2d.astype(float).values
        df[f"of_{key}"] = np.degrees(k1.preferred_OF_closedloop_g2d.astype(float).values)
        df[f"ratio_{key}"] = df[f"rs_{key}"] / np.radians(df[f"of_{key}"])
        df[f"r2_{key}"] = k5.rsof_test_rsq_closedloop_g2d.astype(float).values
    return df


def on_bounds(df, key, param_range):
    """Preferred RS or OF on a bound of `param_range`: the Gaussian is a ramp there."""
    rs, of = df[f"rs_{key}"], df[f"of_{key}"]
    on = lambda x, b: np.isclose(x, b, rtol=1e-3)
    return (
        on(rs, param_range["rs_min"]) | on(rs, param_range["rs_max"])
        | on(of, param_range["of_min"]) | on(of, param_range["of_max"])
    )


fits = pd.concat([load_fits(s) for s in SESSIONS], ignore_index=True)
fits["mouse"] = fits.session.str.split("_").str[0]
for key in FITS:
    fits[f"bound_{key}"] = on_bounds(fits, key, TIGHT if key.startswith("T") else WIDE)
fits.to_csv(save_folder / "rsof_fits_spikes_vs_dff.csv", index=False)
fits.groupby("session")[[f"r2_{k}" for k in FITS]].quantile(0.95).round(3)

Spike fits explain less variance per frame than dF/F fits (single-frame spikes are noisy), so a threshold
on the free-locomotion R² would keep few neurons and bias the selection. The selection uses the treadmill only.

In [ ]:
def select(df, r2_thr=R2_THR, unbound_free=False):
    sel = df.is_depth_neuron_treadmill & (df.r2_T > r2_thr) & ~df.bound_T
    if unbound_free:
        sel &= ~df.bound_A & ~df.bound_B
    return df[sel]


base = fits.is_depth_neuron_treadmill & (fits.r2_T > R2_THR)
print(f"depth tuned on treadmill with T test R² > {R2_THR}: {base.sum()}")
print(f"  of which T on the tight bounds (excluded): {(base & fits.bound_T).sum()}")
sel = select(fits)
print(f"selected: {len(sel)}")
print(f"  A on the wide bounds: {sel.bound_A.sum()}, B: {sel.bound_B.sum()}, either: {(sel.bound_A | sel.bound_B).sum()}")
print(f"  T' on the tight bounds: {sel.bound_Tp.sum()}")

## Distance to the treadmill fit

For each ROI, distance = |log10 X − log10 T| with X = A or B. Paired Wilcoxon on the distances.
Bias = median of log10 X − log10 T. T′ − T shows how much deconvolution changes the treadmill fit itself.

In [ ]:
def compare(df):
    rows = []
    for var in ["rs", "of", "ratio"]:
        lt, la, lb, ltp = (np.log10(df[f"{var}_{k}"]) for k in ["T", "A", "B", "Tp"])
        dA, dB = np.abs(la - lt), np.abs(lb - lt)
        rows.append(dict(
            var=var,
            n=len(df),
            dist_A=dA.median(),
            dist_B=dB.median(),
            frac_B_closer=np.mean(dB < dA),
            p_wilcoxon=stats.wilcoxon(dA, dB).pvalue if len(df) > 1 else np.nan,
            r_AT=stats.pearsonr(la, lt)[0],
            r_BT=stats.pearsonr(lb, lt)[0],
            r_AB=stats.pearsonr(la, lb)[0],
            bias_A=np.median(la - lt),
            bias_B=np.median(lb - lt),
            dist_Tp=np.median(np.abs(ltp - lt)),
        ))
    return pd.DataFrame(rows).set_index("var")


def compare_by_session(df):
    out = {"all": compare(df)}
    for session, sdf in df.groupby("session"):
        out[session] = compare(sdf)
    return pd.concat(out, names=["session"])


results = {
    "selected": compare_by_session(select(fits)),
    "A and B not on bounds": compare_by_session(select(fits, unbound_free=True)),
}
results = pd.concat(results, names=["subset"])
results.to_csv(save_folder / "distance_to_treadmill.csv")
results.round(3)

### Threshold sweep

In [ ]:
sweep = []
for thr in [0, 0.01, 0.02, 0.05, 0.1, 0.2]:
    for unbound in [False, True]:
        df = select(fits, r2_thr=thr, unbound_free=unbound)
        if len(df) < 5:
            continue
        res = compare(df)
        res["r2_thr"], res["unbound_free"] = thr, unbound
        sweep.append(res.reset_index())
sweep = pd.DataFrame(pd.concat(sweep)).set_index(["unbound_free", "r2_thr", "var"])
sweep[["n", "dist_A", "dist_B", "frac_B_closer", "p_wilcoxon", "bias_A", "bias_B"]].round(3)

## Figures

In [ ]:
VARS = {
    "rs": ("Preferred RS (cm/s)", 100),
    "of": ("Preferred OF (°/s)", 1),
    "ratio": ("RS/OF (cm)", 100),
}
SESSION_COLORS = dict(zip(SESSIONS, plt.get_cmap("tab10").colors))


def scatter_vs_treadmill(df, title):
    fig, axes = plt.subplots(2, 3, figsize=(18 * cm, 12 * cm), layout="constrained")
    for icol, (var, (label, scale)) in enumerate(VARS.items()):
        for irow, key in enumerate(["A", "B"]):
            ax = axes[irow, icol]
            for session, sdf in df.groupby("session"):
                ax.scatter(
                    sdf[f"{var}_T"] * scale, sdf[f"{var}_{key}"] * scale, s=6, alpha=0.7,
                    color=SESSION_COLORS[session], label=session.split("_")[0], lw=0,
                )
            ax.set_xscale("log")
            ax.set_yscale("log")
            lims = np.array([ax.get_xlim(), ax.get_ylim()])
            lims = [lims[:, 0].min(), lims[:, 1].max()]
            ax.plot(lims, lims, color="grey", ls=":", lw=0.8, zorder=-1)
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            ax.set_aspect("equal")
            r = stats.pearsonr(np.log10(df[f"{var}_T"]), np.log10(df[f"{var}_{key}"]))[0]
            ax.set_title(f"{LABELS[key]}, r={r:.2f}", fontsize=8)
            ax.set_xlabel(f"{label}\n{LABELS['T']}", fontsize=7)
            ax.set_ylabel(f"{label}\n{LABELS[key]}", fontsize=7)
            ax.tick_params(labelsize=6)
    axes[0, 0].legend(fontsize=5, markerscale=1.5, frameon=False)
    fig.suptitle(f"{title} (n={len(df)})", fontsize=9)
    return fig


for name, unbound in [("selected", False), ("A and B not on bounds", True)]:
    fig = scatter_vs_treadmill(select(fits, unbound_free=unbound), name)
    fig.savefig(save_folder / f"scatter_vs_treadmill_{name.replace(' ', '_')}.pdf")

In [ ]:
def plot_bias(df, title):
    fig, axes = plt.subplots(2, 3, figsize=(18 * cm, 10 * cm), layout="constrained")
    for icol, (var, (label, _)) in enumerate(VARS.items()):
        lt, la, lb = (np.log10(df[f"{var}_{k}"]) for k in ["T", "A", "B"])
        # signed bias relative to the treadmill
        ax = axes[0, icol]
        bins = np.linspace(-3, 3, 41)
        ax.hist(la - lt, bins=bins, histtype="step", label=LABELS["A"])
        ax.hist(lb - lt, bins=bins, histtype="step", label=LABELS["B"])
        ax.axvline(0, color="grey", ls=":")
        ax.set_xlabel(f"log10 {label.split(' (')[0]} − treadmill", fontsize=7)
        # paired distances
        ax = axes[1, icol]
        dA, dB = np.abs(la - lt), np.abs(lb - lt)
        ax.scatter(dA, dB, s=5, alpha=0.6, lw=0)
        m = max(dA.max(), dB.max())
        ax.plot([0, m], [0, m], color="grey", ls=":")
        ax.set_aspect("equal")
        ax.set_xlabel("|A − T| (log10)", fontsize=7)
        ax.set_ylabel("|B − T| (log10)", fontsize=7)
        ax.set_title(f"B closer: {np.mean(dB < dA):.0%}, p={stats.wilcoxon(dA, dB).pvalue:.2g}", fontsize=7)
        for a in axes[:, icol]:
            a.tick_params(labelsize=6)
    axes[0, 0].legend(fontsize=6, frameon=False)
    fig.suptitle(f"{title} (n={len(df)})", fontsize=9)
    return fig


for name, unbound in [("selected", False), ("A and B not on bounds", True)]:
    fig = plot_bias(select(fits, unbound_free=unbound), name)
    fig.savefig(save_folder / f"bias_vs_treadmill_{name.replace(' ', '_')}.pdf")

In [ ]:
# Per session: median distance to the treadmill, A vs B
fig, axes = plt.subplots(1, 3, figsize=(15 * cm, 5 * cm), layout="constrained")
res = results.loc["A and B not on bounds"].drop(index="all", level="session")
for ax, var in zip(axes, VARS):
    r = res.xs(var, level="var")
    for session, row in r.iterrows():
        ax.plot([0, 1], [row.dist_A, row.dist_B], "o-", color=SESSION_COLORS[session],
                label=f"{session.split('_')[0]} (n={row.n:.0f})")
    ax.set_xticks([0, 1], ["dF/F", "spikes"])
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(0, None)
    ax.set_title(VARS[var][0].split(" (")[0], fontsize=8)
    ax.tick_params(labelsize=6)
axes[0].set_ylabel("median |log10 free − treadmill|", fontsize=7)
axes[-1].legend(fontsize=5, frameon=False, loc="lower left")
fig.savefig(save_folder / "distance_per_session.pdf")

## Example cells: RS/OF matrices

Mean response in RS/OF bins (data, not fits) for the four conditions, on the treadmill grid
(`compute_treadmill_rsof_bins`, as in the treadmill figure). Per-frame data, running frames, closed loop
only; treadmill frames with |RS − motor| / RS ≥ 0.3 are excluded. Each panel has its own colour scale.
Markers: preferred RS/OF of the fit of that panel (○) and of the treadmill dF/F fit (+). Markers outside the grid
are not shown.

Saved to `example_cells/`: one PNG and one `.npz` for each of the `N_EXAMPLE_CELLS` selected ROIs
with the best free-locomotion fits (min of dF/F and spike test R²) (matrices `free_dff`, `free_spks`,
`treadmill_dff`, `treadmill_spks`, bin edges in cm/s and °/s; the first row and column are the bins below
the grid), plus `example_cells.csv` with the fit values.

In [ ]:
example_folder = save_folder / "example_cells"
example_folder.mkdir(exist_ok=True)
MATRICES = {  # name: (protocol, response, fit drawn on the panel)
    "free_dff": ("free", "dff", "A"),
    "free_spks": ("free", "spks", "B"),
    "treadmill_dff": ("treadmill", "dff", "T"),
    "treadmill_spks": ("treadmill", "spks", "Tp"),
}


def load_trials(session, protocol_base):
    _, _, _, trials_df = pipeline_utils.load_session(
        project=PROJECT,
        session_name=session,
        photodiode_protocol=frt.PHOTODIODE_PROTOCOL,
        regenerate_frames=False,
        filter_datasets=frt.FILTER_DATASETS,
        protocol_base=protocol_base,
        recording_type="two_photon",
        tread_kwargs={"method": "plateau"},
        add_spikes=True,
    )
    # the fits exclude multidepth recordings
    trials_df = trials_df[~trials_df.recording_name.str.contains("multidepth")]
    return trials_df.reset_index(drop=True)


def plot_example(row, trials, rs_bins, of_bins, tick_dict):
    fig, axes = plt.subplots(1, 4, figsize=(26 * cm, 7 * cm))
    matrices = {}
    for ax, (name, (protocol, resp, key)) in zip(axes, MATRICES.items()):
        tdf = trials[protocol]
        if resp == "spks":
            tdf = tdf.assign(dff_stim=tdf.spks_stim)
        _, _, matrices[name] = rsof_plots.plot_RS_OF_matrix(
            trials_df=tdf,
            roi=row.roi,
            is_closed_loop=1,
            max_abs_rs2motor_diff_ratio=0.3,
            ax=ax,
            log_range={"log_base": 2},
            rs_bins=rs_bins,
            of_bins=of_bins,
            tick_dict=tick_dict,
            title=f"{LABELS[key]}\nR²={row[f'r2_{key}']:.2f}",
            fontsize_dict={"title": 7, "label": 7, "tick": 6, "legend": 5},
            return_matrix=True,
        )
        if resp == "spks":  # the colour bar is the last axes added
            fig.axes[-1].set_ylabel("spikes", fontsize=6)
        # preferences outside the grid are not drawn, without changing the limits
        xlim, ylim = ax.get_xlim(), ax.get_ylim()
        for k, marker in [(key, "o"), ("T", "+")]:
            ax.plot(np.log2(row[f"rs_{k}"] * 100), np.log2(row[f"of_{k}"]), marker,
                    color="k", mfc="none", ms=6, clip_on=True)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        if ax is not axes[0]:
            ax.set_ylabel("")
    fig.suptitle(f"{row.session} ROI {row.roi}", fontsize=8, y=1.08)
    return fig, matrices


N_EXAMPLE_CELLS = 15
# selected cells with the best free-locomotion fits in both dF/F and spikes
examples = (
    select(fits)
    .assign(r2_free=lambda x: np.minimum(x.r2_A, x.r2_B))
    .nlargest(N_EXAMPLE_CELLS, "r2_free")
)
examples.to_csv(example_folder / "example_cells.csv", index=False)
for session in SESSIONS:
    rois = examples[examples.session == session]
    print(f"{session}: {len(rois)} ROIs")
    if rois.empty:
        continue
    trials = {
        "free": load_trials(session, "SpheresPermTubeReward"),
        "treadmill": load_trials(session, "SpheresTubeMotor"),
    }
    rs_bins, of_bins, tick_dict = compute_treadmill_rsof_bins(trials["treadmill"])
    for _, row in rois.iterrows():
        fig, matrices = plot_example(row, trials, rs_bins, of_bins, tick_dict)
        # session names contain dots, so build file names rather than use with_suffix
        stem = f"{session}_roi{row.roi:04d}"
        fig.savefig(example_folder / f"{stem}.png", dpi=150, bbox_inches="tight")
        plt.close(fig)
        np.savez(
            example_folder / f"{stem}.npz",
            rs_bins_cm_s=rs_bins,
            of_bins_deg_s=of_bins,
            **matrices,
            **{f"{v}_{k}": row[f"{v}_{k}"] for v in ["rs", "of", "r2"] for k in FITS},
        )
    del trials
print(f"Saved {len(examples)} example cells to {example_folder}")

## Response lag (removed 2026-10-20)

A response-lag analysis was explored here in October 2026 and then dropped: the manuscript RS/OF fits pair
RS/OF and response of the same imaging frame (lag 0), and the lag sweep scripts and their temporary
results were deleted. What it found, for the record:

- Free-locomotion `gaussian_2d` refits of every ROI of the four motor sessions with the response read
  −5 to +30 imaging frames after RS/OF. Best mean test R² (normalised per cell): spikes 0.33 s (GCaMP6s)
  and 0.20 s (GCaMP6f); dF/F 0.99 s (GCaMP6s, flat plateau) and 0.33 s (GCaMP6f). dF/F lagged spikes in
  almost every cell, as expected from the indicator kernels (mean delay ≈0.84 s GCaMP6s, ≈0.19 s GCaMP6f).
- At these lags, fits became less elongated and closer to the treadmill fit in preferred RS, OF, angle and
  elongation (most with spikes; e.g. median |log10 RS − log10 T| 0.87 → 0.42, n = 92), but not in
  preferred RS/OF (0.20 → 0.18, p = 0.76). Most fits stayed diagonal (spikes 68% vs dF/F lag 0 85%;
  treadmill 35%).